# NB_hospital_daily_summary

Build `adb_caresync.gold.hospital_daily_summary` from the STTM workbook.

* Target grain: one row per hospital per calendar day
* Driving table: aggregated `adb_caresync.silver.lab_results`
* Required join: `adb_caresync.silver.hospitals`
* Optional date enrichment: `adb_caresync.silver.calendar`
* Load type: full rebuild with `CREATE OR REPLACE TABLE`
* Verified schema adjustments from source inspection:
  * the calendar table uses `datekey`, not `date_key`, so this notebook joins on `calendar.datekey = CAST(date_format(summary_date, 'yyyyMMdd') AS INT)`
  * all lab-result fields required for counts and window logic exist in `adb_caresync.silver.lab_results`


In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary AS
WITH daily_aggregates AS (
  SELECT
    hospital_id,
    test_date AS summary_date,
    COUNT(*) AS total_tests,
    COUNT(DISTINCT patient_id) AS unique_patients_tested,
    COUNT(DISTINCT doctor_id) AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY hospital_id, test_date
),
enriched AS (
  SELECT
    da.hospital_id,
    h.hospital_name,
    da.summary_date,
    c.day_name,
    c.is_weekend,
    da.total_tests,
    da.unique_patients_tested,
    da.active_doctor_count,
    da.abnormal_test_count,
    da.critical_test_count,
    CAST(ROUND(CASE WHEN da.total_tests = 0 THEN NULL ELSE da.abnormal_test_count * 100.0 / da.total_tests END, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(CASE WHEN da.total_tests = 0 THEN NULL ELSE da.critical_test_count * 100.0 / da.total_tests END, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM daily_aggregates da
  LEFT JOIN adb_caresync.silver.hospitals h
    ON da.hospital_id = h.hospital_id
  LEFT JOIN adb_caresync.silver.calendar c
    ON c.datekey = CAST(date_format(da.summary_date, 'yyyyMMdd') AS INT)
)
SELECT
  hospital_id,
  hospital_name,
  summary_date,
  day_name,
  is_weekend,
  total_tests,
  unique_patients_tested,
  active_doctor_count,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests,
  CAST(
    ROUND(
      CASE
        WHEN LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) IS NULL
          OR LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) = 0 THEN NULL
        ELSE (total_tests - LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date))
          * 100.0
          / LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date)
      END,
      2
    ) AS DECIMAL(6,2)
  ) AS test_volume_change_pct
FROM enriched;

In [0]:
%sql
WITH source_counts AS (
  SELECT COUNT(*) AS expected_rows
  FROM (
    SELECT hospital_id, test_date
    FROM adb_caresync.silver.lab_results
    GROUP BY hospital_id, test_date
  ) x
),
gold_stats AS (
  SELECT
    COUNT(*) AS gold_rows,
    MIN(summary_date) AS min_summary_date,
    MAX(summary_date) AS max_summary_date,
    SUM(CASE WHEN prev_day_total_tests IS NULL THEN 1 ELSE 0 END) AS first_day_rows,
    SUM(CASE WHEN day_name IS NULL THEN 1 ELSE 0 END) AS rows_missing_calendar_enrichment
  FROM adb_caresync.gold.hospital_daily_summary
)
SELECT
  sc.expected_rows,
  gs.gold_rows,
  sc.expected_rows = gs.gold_rows AS row_count_matches,
  gs.min_summary_date,
  gs.max_summary_date,
  gs.first_day_rows,
  gs.rows_missing_calendar_enrichment
FROM source_counts sc
CROSS JOIN gold_stats gs;